In [ ]:
# CELL 1 - Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers transformers accelerate sentencepiece
!pip install -q fastapi uvicorn pyngrok "imageio[ffmpeg]"
!pip install -q opencv-python pillow numpy huggingface_hub
print("All dependencies installed!")

In [ ]:
# CELL 2 - Download LTX-Video model
import os
from huggingface_hub import snapshot_download
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except:
    hf_token = os.environ.get("HF_TOKEN", "__HF_TOKEN_PLACEHOLDER__")

print("Downloading LTX-Video model...")
model_id = "Lightricks/LTX-Video"
local_dir = "/kaggle/working/ltx-model"
if hf_token:
    snapshot_download(repo_id=model_id, local_dir=local_dir, token=hf_token)
else:
    snapshot_download(repo_id=model_id, local_dir=local_dir)
print("LTX-Video model downloaded!")

In [ ]:
# CELL 3 - Load LTX model and start FastAPI server
import threading, uvicorn, os, torch, time, base64
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from diffusers import LTXVideoPipeline
from diffusers.utils import export_to_video

print("Loading LTX-Video pipeline on GPU...")
pipe = LTXVideoPipeline.from_pretrained("/kaggle/working/ltx-model", torch_dtype=torch.bfloat16)
pipe = pipe.to("cuda")
print("LTX-Video ready on GPU!")

app = FastAPI()
last_activity = time.time()
IDLE_TIMEOUT = 10 * 60

class GenerateRequest(BaseModel):
    prompt: str
    scene_id: str
    account_id: str

@app.post("/generate")
def generate_video(request: GenerateRequest):
    global last_activity
    last_activity = time.time()
    print(f"Generating: {request.prompt}")
    try:
        result = pipe(
            prompt=request.prompt,
            negative_prompt="worst quality, blurry, jittery",
            width=512, height=288, num_frames=49,
            num_inference_steps=30,
        )
        frames = result.frames[0]
        output_file = f"/kaggle/working/{request.scene_id}.mp4"
        export_to_video(frames, output_file, fps=24)
        with open(output_file, "rb") as f:
            video_b64 = base64.b64encode(f.read()).decode()
        last_activity = time.time()
        return {"status": "success", "video_path": output_file, "video_b64": video_b64}
    except Exception as e:
        return JSONResponse(status_code=500, content={"status": "error", "message": str(e)})

@app.get("/health")
def health():
    return {"status": "alive", "model": "LTX-Video"}

def idle_checker():
    global last_activity
    while True:
        time.sleep(30)
        if time.time() - last_activity > IDLE_TIMEOUT:
            print("Idle 10 min. Shutting down...")
            os._exit(0)

threading.Thread(target=idle_checker, daemon=True).start()
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
print("FastAPI server ready on port 8000!")

In [ ]:
# CELL 4 - Start ngrok tunnel & Register with backend
import os, requests
from pyngrok import ngrok
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    ngrok_token = user_secrets.get_secret("NGROK_TOKEN")
except:
    ngrok_token = os.environ.get("NGROK_TOKEN", "__NGROK_TOKEN_PLACEHOLDER__")

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    kaggle_user = user_secrets.get_secret("KAGGLE_USERNAME")
except:
    kaggle_user = os.environ.get("KAGGLE_USERNAME", "__KAGGLE_USERNAME_PLACEHOLDER__")

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8000).public_url
print(f"Ngrok URL: {public_url}")

backend_url = "https://irfangull2288--cartoon-backend-fastapi-modal-app.modal.run/session/register-url"
try:
    resp = requests.post(backend_url, json={"account_username": kaggle_user, "url": public_url})
    print("Registered:", resp.json())
except Exception as e:
    print("Registration failed:", e)

In [ ]:
# CELL 5 - Keep alive
import time
print("System fully ready! Waiting for generation requests...")
while True:
    time.sleep(60)